# Adagrad 详解：历史平方和 · 自适应缩放 · 利用梯度大小


> **引子：从 Rprop 的遗憾说起**
>
> 在优化算法的演进史中，Rprop（1993）率先提出了一个革命性理念：**每个参数应该拥有自己独立的步长**。它不再像梯度下降那样对所有参数“一刀切”，而是根据梯度的符号变化，动态调整每个维度的步长。
>
> 然而，Rprop 有一个令人遗憾的短板——**它完全丢弃了梯度的大小信息**。它只关心“梯度是正还是负”，却对“梯度有多大”视而不见。梯度大小本身是极有价值的信号：梯度大，说明该方向敏感，步长应该小；梯度小，说明该方向平缓，步长应该大。Rprop 的这种“一刀切”式忽略，导致它的步长调节只能依赖符号反转来**事后刹车**——等已经跨过了谷底，才知道步子迈大了。
>
> **Adagrad（2011）正是为了解决这个遗憾而生的**。它继承了 Rprop 的核心理念——每个参数独立步长，但做了一个关键改进：**用历史梯度平方的累加来直接缩放学习率**。这一改，让梯度大小从“被丢弃的废料”变成了“自适应的指针”，实现了从“事后刹车”到“事前缩放”的进化。
>
> 本讲将带你深入 Adagrad 的设计机理、迭代规则、实验验证，并揭示它为何在稀疏数据上大放异彩，又为何因学习率单调递减而留下遗憾——这个遗憾，恰恰催生了后面的 RMSprop 和 Adam。
>
> **一句话**：Adagrad 让每个参数都学会“看梯度大小自动调步长”，代价是它永远在缩小——这既成就了它的自适应能力，也埋下了“学习率死亡”的隐患。

| 项目 | 内容 |
|------|------|
| **全称** | **Adaptive Gradient Algorithm**（自适应梯度算法） |
| **中文译名** | 自适应梯度算法 / Adagrad |
| **提出者** | John Duchi, Elad Hazan, Yoram Singer |
| **提出年份** | 2011 年 |
| **发表会议** | COLT 2011 (Journal of Machine Learning Research 2011 正式版) |
| **学术简称** | Adagrad |
| **深度学习社区常用名** | Adagrad（无别名） |
| **所属家族** | 自适应学习率方法（每个参数独立学习率） |
| **直接前身** | **Rprop**（继承"每个参数独立步长"理念，改用梯度大小累加替代符号一致性） |
| **核心创新** | 用历史梯度平方和累加来缩放每个参数的学习率——梯度大的参数学习率小，梯度小的参数学习率大 |
| **理论收敛性** | 对凸函数有理论收敛保证；在非凸情况下能收敛到临界点（梯度为零的点） |
| **关于鞍点的能力边界** | Adagrad 具有一定绕过鞍点的能力（利用梯度大小累加持续提供推进力），但这种能力**仅仅意味着避开鞍点**，**绝不意味着找到更优点**，同时也意味着**不保证收敛到最近的极小值点**——它只是“离开当前点”，至于下一站是更优、更差还是同等水平，算法自身完全无法判断 |
| **致命局限** | 学习率**单调递减**，训练后期学习率趋近于零，模型**强制停滞** |

> 对应原文附录术语对照表：**Adagrad | Adagrad | Duchi et al., 2011**

## 1 快速感知 Adagrad 的过人之处

在深入 Adagrad 的数学机理之前，我们先通过两个实验，直观感受 Adagrad 相比 Rprop 的进步。
### 1.1 算例一：逃离鞍点

**目标**：在一个具有局部最优、鞍点和全局最优的多模态函数上，对比 Adagrad 与 Rprop 逃离鞍点的能力。

> **关键说明**：本算例展示的是 Adagrad **具有一定绕过鞍点的能力**这一事实。但需要明确以下三点：
> 1. **这种能力仅仅意味着避开鞍点**，即 Adagrad 不会像 Rprop 那样因符号抖动而在鞍点处完全停滞。
> 2. **绝不意味着找到更优点**——绕过鞍点后，算法可能落入更深的极小值（更好），也可能落入更浅的极小值（更差），甚至落入另一个鞍点（平级）。
> 3. **不保证收敛到最近的极小值点**——Adagrad 可能“穿过”一个较近的浅谷，跑到更远的区域去。
>
> 至于绕过后去向何方，取决于具体的地形和参数设置，算法自身没有任何判断机制。

In [ ]:
# ============================================================
# 代码块1：导入必要的库并配置全局参数
# ============================================================

import numpy as np
import pandas as pd
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import warnings
warnings.filterwarnings('ignore')

# ============================================================
# 第一部分：全局参数配置（集中设置）
# ============================================================

# ---------- 目标函数参数 ----------
FUNCTION_PARAMS = {
    'a': 0.1,
    'b': 10.0,
}

# ---------- 统一迭代参数 ----------
ITERATION_PARAMS = {
    'max_iter': 50,
}

# ---------- Rprop 算法参数 ----------
RPROP_PARAMS = {
    'alpha_plus': 1.2,
    'alpha_minus': 0.5,
    'delta_max': 50.0,
    'delta_min': 1e-6,
    'delta_init': 0.1,
}

# ---------- Adagrad 算法参数 ----------
ADAGRAD_PARAMS = {
    'lr': 0.5,
    'epsilon': 1e-8,
}

# ---------- 初始点参数 ----------
INIT_POINT = {
    'theta0': np.array([1.0, 0.1]),
}

# ---------- 绘图参数 ----------
PLOT_PARAMS = {
    'contour_range': (-1.5, 1.5),
    'contour_points': 200,
    'width': 1400,
    'height_contour': 650,
    'height_standard': 450,
}

# ---------- 图例样式 ----------
LEGEND_STYLE = dict(
    x=0.5,
    y=1.15,
    xanchor='center',
    yanchor='bottom',
    orientation='h',
    bgcolor='rgba(255,255,255,0)',
    font=dict(size=12),
    bordercolor='rgba(255,255,255,0)',
    borderwidth=0
)

# ---------- 图形边距设置 ----------
MARGIN_STYLE = dict(l=80, r=40, t=100, b=60)

# ============================================================
# 代码块2：算例一 —— 环境配置与函数定义
# ============================================================

# %%
"""
模块1：环境配置与函数定义
"""
import numpy as np
import pandas as pd
import plotly.graph_objects as go

# 全局配置字典
CONFIG = {
    "start_x": -1.3,
    "start_y": 0.2,
    "steps": 50,
    "rprop_eta_plus": 1.2,
    "rprop_eta_minus": 0.5,
    "rprop_delta_min": 1e-6,
    "rprop_delta_max": 10.0,
    "rprop_delta_init": 0.005,
    "adagrad_lr": 1.5,
    "adagrad_eps": 1e-8,
    "verbose": True,
    "max_print": 10000,
    "x_min": -1.8, "x_max": 1.6, "y_min": -1.6, "y_max": 0.6,
    "mesh_res": 300,
    "fig_height": 700, "fig_width": None,
    "path_line_width": 2.0, "path_marker_size": 5,
}

# 定义关键点坐标
local_min = (-0.8882, 0.0)
saddle_pt = (-0.1949, 0.0)
global_min = (1.0831, 0.0)

def f(x, y):
    """
    多模态目标函数：具有局部最优、鞍点和全局最优
    f(x,y) = 0.2 * (x² - 1)² + y² - 0.15 * x
    """
    return 0.2 * (x**2 - 1)**2 + y**2 - 0.15 * x

def grad_f(x, y):
    """
    目标函数的梯度
    ∂f/∂x = 0.8 * x * (x² - 1) - 0.15
    ∂f/∂y = 2 * y
    """
    dx = 0.8 * x * (x**2 - 1) - 0.15
    dy = 2 * y
    return dx, dy

# ============================================================
# 代码块3：算例一 —— Rprop 算法实现与运行
# ============================================================

# %%
"""
模块2：Rprop 算法实现与运行
"""
def rprop_optimizer(start_x, start_y, eta_plus, eta_minus, delta_min, delta_max, delta_init, steps, max_print):
    x, y = start_x, start_y
    delta_x, delta_y = delta_init, delta_init
    prev_gx, prev_gy = 0.0, 0.0
    path = [(x, y, f(x, y))]
    
    data = []
    data.append([0, x, y, np.nan, np.nan, delta_x, delta_y, x, y, f(x, y)])

    for t in range(steps):
        gx, gy = grad_f(x, y)
        
        if t > 0:
            if gx * prev_gx > 0:
                delta_x = min(delta_x * eta_plus, delta_max)
            elif gx * prev_gx < 0:
                delta_x = max(delta_x * eta_minus, delta_min)
            if gy * prev_gy > 0:
                delta_y = min(delta_y * eta_plus, delta_max)
            elif gy * prev_gy < 0:
                delta_y = max(delta_y * eta_minus, delta_min)
        
        x_new = x - np.sign(gx) * delta_x if gx != 0 else x
        y_new = y - np.sign(gy) * delta_y if gy != 0 else y
        
        if t < max_print:
            data.append([t+1, x, y, gx, gy, delta_x, delta_y, x_new, y_new, f(x_new, y_new)])
        
        x, y = x_new, y_new
        prev_gx, prev_gy = gx, gy
        path.append((x, y, f(x, y)))
    
    df = pd.DataFrame(data, columns=['步数', 'x_t', 'y_t', '∂f/∂x', '∂f/∂y', 'delta_x', 'delta_y', 'x_{t+1}', 'y_{t+1}', 'f(x,y)'])
    print("【Rprop 详细迭代过程】")
    print(f"eta_plus = {eta_plus}, eta_minus = {eta_minus}, delta_init = {delta_init}")
    display(df)
    
    return np.array(path)

start_x = CONFIG["start_x"]
start_y = CONFIG["start_y"]
common_steps = CONFIG["steps"]

path_rprop = rprop_optimizer(
    start_x=start_x, start_y=start_y,
    eta_plus=CONFIG["rprop_eta_plus"], eta_minus=CONFIG["rprop_eta_minus"],
    delta_min=CONFIG["rprop_delta_min"], delta_max=CONFIG["rprop_delta_max"],
    delta_init=CONFIG["rprop_delta_init"],
    steps=common_steps, max_print=CONFIG["max_print"]
)


In [ ]:
# ============================================================
# 代码块4：算例一 —— Adagrad 算法实现与运行
# ============================================================

# %%
"""
模块3：Adagrad 算法实现与运行
"""
def adagrad_optimizer(start_x, start_y, lr, eps, steps, max_print):
    x, y = start_x, start_y
    G_x, G_y = 0.0, 0.0
    path = [(x, y, f(x, y))]
    
    data = []
    data.append([0, x, y, np.nan, np.nan, G_x, G_y, x, y, f(x, y)])

    for t in range(steps):
        gx, gy = grad_f(x, y)
        
        G_x += gx ** 2
        G_y += gy ** 2
        
        x_new = x - lr * gx / (np.sqrt(G_x) + eps)
        y_new = y - lr * gy / (np.sqrt(G_y) + eps)
        
        if t < max_print:
            data.append([t+1, x, y, gx, gy, G_x, G_y, x_new, y_new, f(x_new, y_new)])
        
        x, y = x_new, y_new
        path.append((x, y, f(x, y)))
    
    df = pd.DataFrame(data, columns=['步数', 'x_t', 'y_t', '∂f/∂x', '∂f/∂y', 'G_x_t', 'G_y_t', 'x_{t+1}', 'y_{t+1}', 'f(x,y)'])
    print("【Adagrad 详细迭代过程】")
    print(f"学习率 α = {lr}, epsilon = {eps}")
    display(df)
    
    return np.array(path)

path_adagrad = adagrad_optimizer(
    start_x=start_x, start_y=start_y,
    lr=CONFIG["adagrad_lr"], eps=CONFIG["adagrad_eps"],
    steps=common_steps, max_print=CONFIG["max_print"]
)


In [ ]:
# ============================================================
# 代码块5：算例一 —— 绘制迭代路径对比图
# ============================================================

# %%
"""
模块4：绘制迭代路径对比图
"""
xs = np.linspace(CONFIG["x_min"], CONFIG["x_max"], CONFIG["mesh_res"])
ys = np.linspace(CONFIG["y_min"], CONFIG["y_max"], CONFIG["mesh_res"])
X, Y = np.meshgrid(xs, ys)
Z = f(X, Y)

fig1 = go.Figure()

fig1.add_trace(go.Contour(
    x=xs, y=ys, z=Z,
    colorscale="Viridis",
    contours=dict(showlabels=True, size=0.05),
    opacity=0.8
))

fig1.add_trace(go.Scatter(
    x=path_rprop[:, 0], y=path_rprop[:, 1],
    mode="lines+markers",
    line=dict(color="red", width=CONFIG["path_line_width"], dash="solid"),
    marker=dict(size=CONFIG["path_marker_size"], color="red", symbol="circle"),
    name="Rprop路径"
))

fig1.add_trace(go.Scatter(
    x=path_adagrad[:, 0], y=path_adagrad[:, 1],
    mode="lines+markers",
    line=dict(color="blue", width=CONFIG["path_line_width"], dash="dot"),
    marker=dict(size=CONFIG["path_marker_size"], color="blue", symbol="square"),
    name="Adagrad路径"
))

fig1.add_trace(go.Scatter(
    x=[start_x], y=[start_y],
    mode="markers",
    marker=dict(size=14, color="#ffcc00", line=dict(color="black", width=1)),
    name="起点"
))

fig1.add_trace(go.Scatter(
    x=[local_min[0]], y=[local_min[1]],
    mode="markers",
    marker=dict(size=12, color="red", symbol="diamond", line=dict(color="black", width=1)),
    name="局部最优"
))

fig1.add_trace(go.Scatter(
    x=[saddle_pt[0]], y=[saddle_pt[1]],
    mode="markers",
    marker=dict(size=12, color="#ffaa00", symbol="diamond-open", line=dict(color="black", width=1)),
    name="鞍点"
))

fig1.add_trace(go.Scatter(
    x=[global_min[0]], y=[global_min[1]],
    mode="markers",
    marker=dict(size=16, color="yellow", symbol="star", line=dict(color="black", width=1)),
    name="全局最优"
))

fig1.update_layout(
    title="图1：Rprop vs Adagrad：迭代路径对比",
    xaxis_title="x",
    yaxis_title="y",
    legend=dict(
        orientation="h",
        yanchor="bottom",
        y=1.02,
        xanchor="center",
        x=0.5,
        font=dict(size=12),
        bordercolor=None,
        borderwidth=0,
    )
)

fig1.show(renderer="notebook")


#### 图1 解读

   - **Rprop 在非凸地形（含鞍点）中表现不佳**——梯度符号的频繁抖动会反复触发步长收缩，导致优化停滞在局部最优附近。
   - **Adagrad 在鞍点区域仍能保持稳定的推进力**，成功绕过鞍点并收敛到全局最优。
   - 这验证了"Adagrad 逃离鞍点能力优于 Rprop"的判断——**梯度大小信息的保留，是在复杂地形中保持动量的关键**。

> **需要强调以下三点**：
> 1. 本算例中 Adagrad "绕过鞍点后找到了全局最优"，这**绝不意味着绕过鞍点必然导致更优点**。在另一个函数或另一组参数下，Adagrad 绕过鞍点后可能落入一个更浅的局部极小值，甚至比鞍点本身更差。
> 2. Adagrad 的这项能力**仅仅意味着避开鞍点**——它不会在鞍点处停滞，仅此而已。
> 3. Adagrad **不保证收敛到最近的极小值点**——它可能“穿过”一个较近的浅谷，跑到更远的地方去。
>
> 这个实验展示的是 Adagrad 的**能力**（能够绕过），而非**保证**（一定找到更优）。

> **一句话总结**：在非凸地形上，Rprop 因梯度符号抖动而"迷路"（被困局部最优），Adagrad 因梯度大小累加而"认路"（具备绕过鞍点的能力）。但这种能力仅仅意味着避开鞍点，绝不意味着找到更优点，也不保证收敛到最近的极小值点。

In [ ]:
# ============================================================
# 代码块6：算例一 —— 绘制收敛曲线图
# ============================================================

# %%
"""
模块5：绘制收敛曲线图
"""
import numpy as np
import plotly.graph_objects as go

loss_rprop = path_rprop[:, 2]
loss_adagrad = path_adagrad[:, 2]
steps_arr = np.arange(len(loss_rprop))
optimal_loss = f(*global_min)

print(f"Adagrad 最终损失: {loss_adagrad[-1]:.10e}")
print(f"Rprop 最终损失: {loss_rprop[-1]:.10e}")
print(f"最低损失: {min(np.min(loss_adagrad), np.min(loss_rprop)):.10e}")

fig2 = go.Figure()

fig2.add_trace(go.Scatter(
    x=steps_arr, y=loss_rprop,
    mode="lines+markers",
    line=dict(color="red", width=1.5, dash="solid"),
    marker=dict(size=5, color="red", symbol="circle"),
    name="Rprop法"
))

fig2.add_trace(go.Scatter(
    x=steps_arr, y=loss_adagrad,
    mode="lines+markers",
    line=dict(color="blue", width=1.5, dash="dot"),
    marker=dict(size=5, color="blue", symbol="square"),
    name="Adagrad法"
))

fig2.add_trace(go.Scatter(
    x=[0, steps_arr[-1]], y=[optimal_loss, optimal_loss],
    mode="lines",
    line=dict(color="green", width=1.0, dash="dash"),
    name="全局最优"
))

fig2.update_layout(
    title=dict(text="图2：Rprop vs Adagrad：迭代收敛曲线对比", font=dict(size=18)),
    xaxis_title="迭代步数",
    yaxis_title="损失值",
    xaxis=dict(
        gridcolor='lightgray',
        zeroline=True,
        zerolinecolor='gray',
        range=[-1, len(steps_arr)]
    ),
    yaxis=dict(
        type='linear',
        gridcolor='lightgray',
        zeroline=True,
        zerolinecolor='gray',
        rangemode='normal',
    ),
    legend=dict(
        orientation="h", yanchor="bottom", y=1.02,
        xanchor="center", x=0.5,
        font=dict(size=12), bordercolor=None, borderwidth=0,
    ),
    margin=dict(l=80, r=40, t=100, b=60),
    width=1400,
    height=500,
)

fig2.show(renderer="notebook")

#### 图2 解读

   - **Rprop 被困局部最优**
   - **Adagrad 成功绕过鞍点**，收敛于更优
   - 但需注意：本算例的"更优"是相对于 Rprop 被困的局部最优而言的。Adagrad 找到的是该函数的全局最优，但这只是该特定函数下的结果，**绝不是普适性保证**——Adagrad 绕过鞍点后可能落入任何类型的临界点（更优、更差或平级）。

> **⚠️ 结果解读**：本算例中 Adagrad 绕过了鞍点并找到了全局最优，但这**绝不构成普适性保证**——"绕过鞍点"仅仅意味着避开鞍点，绝不意味着找到更优点，Adagrad 也**不保证收敛到最近的极小值**。该结果仅说明在当前函数和参数设置下 Adagrad 的表现优于 Rprop，不代表任何通用性承诺。
>
> 事实上，在工程实践中我们本就不知道全局最优在何处，因此"不保证收敛到最近极小值"在深度学习工程中并非致命问题——我们只期望算法能持续前进，避免在鞍点处完全停滞即可。

In [ ]:
# ============================================================
# 代码块7：算例一 —— 最终结果分析
# ============================================================

# %%
"""
模块6：最终结果分析
"""
print("\n" + "=" * 65)
print("【逃离鞍点能力分析】")
print("=" * 65)
print(f"起点: ({start_x:.1f}, {start_y:.1f}), 初始损失 = {f(start_x, start_y):.4f}")
print(f"Rprop终点: ({path_rprop[-1,0]:.4f}, {path_rprop[-1,1]:.4f}), 损失 = {loss_rprop[-1]:.4f}")
print(f"Adagrad终点: ({path_adagrad[-1,0]:.4f}, {path_adagrad[-1,1]:.4f}), 损失 = {loss_adagrad[-1]:.4f}")
print(f"全局最优: ({global_min[0]:.4f}, {global_min[1]:.4f}), 损失 = {optimal_loss:.4f}")
print(f"  Adagrad是否越过鞍点 (x > -0.1949): {'是' if path_adagrad[-1, 0] > saddle_pt[0] else '否'}")
print("=" * 65)


### 1.2 算例二：二维狭长山谷对比实验

**目标**：在 $f(\theta) = 0.1\theta_1^2 + 10\theta_2^2$ 上，对比 Adagrad、Rprop 和标准梯度下降（GD）的优化轨迹。该算例重点展示 Adagrad 如何利用"历史梯度平方累加"实现每个维度的自适应学习率，同时暴露其"学习率单调递减"的固有缺陷。

In [ ]:
# ============================================================
# 代码块8：算例二 —— 目标函数定义
# ============================================================

def f_theta(theta):
    a = FUNCTION_PARAMS['a']
    b = FUNCTION_PARAMS['b']
    return a * theta[0]**2 + b * theta[1]**2


def grad_f_theta(theta):
    a = FUNCTION_PARAMS['a']
    b = FUNCTION_PARAMS['b']
    return np.array([2 * a * theta[0], 2 * b * theta[1]])


In [ ]:
# ============================================================
# 代码块9：算例二 —— Rprop 算法实现
# ============================================================

def rprop_irprop(theta0, max_iter=None, verbose=False):
    if max_iter is None:
        max_iter = ITERATION_PARAMS['max_iter']
    
    alpha_plus = RPROP_PARAMS['alpha_plus']
    alpha_minus = RPROP_PARAMS['alpha_minus']
    delta_max = RPROP_PARAMS['delta_max']
    delta_min = RPROP_PARAMS['delta_min']
    delta_init = RPROP_PARAMS['delta_init']
    
    theta = theta0.copy()
    delta = np.array([delta_init, delta_init])
    g_prev = grad_f_theta(theta)
    
    history = [theta.copy()]
    delta_history = [delta.copy()]
    loss_history = [f_theta(theta)]
    
    details = []
    
    details.append({
        'iter': 0,
        'theta1': theta[0], 'theta2': theta[1],
        'g1': g_prev[0], 'g2': g_prev[1],
        'sign1': int(np.sign(g_prev[0])), 'sign2': int(np.sign(g_prev[1])),
        'delta1': delta[0], 'delta2': delta[1],
        'event1': '初始', 'event2': '初始',
        'theta1_after': theta[0], 'theta2_after': theta[1],
        'delta1_after': delta[0], 'delta2_after': delta[1],
        'loss': f_theta(theta)
    })

    for t in range(1, max_iter):
        g = grad_f_theta(theta)
        theta_old = theta.copy()
        delta_old = delta.copy()
        event1, event2 = '', ''
        
        for i in range(len(theta)):
            sign_product = g[i] * g_prev[i]
            
            if g[i] * g_prev[i] > 0:
                delta[i] = min(delta[i] * alpha_plus, delta_max)
                event = f'油门×{alpha_plus:.1f}'
                if i == 0: event1 = event
                else: event2 = event
                
            elif g[i] * g_prev[i] < 0:
                delta[i] = max(delta[i] * alpha_minus, delta_min)
                event = f'刹车×{alpha_minus:.1f}+回退'
                
                theta[i] = theta[i] - np.sign(g[i]) * delta[i]
                
                if i == 0: event1 = event
                else: event2 = event
                
                g[i] = 0
                
            else:
                if i == 0: event1 = '保持'
                else: event2 = '保持'
        
        theta_before_update = theta.copy()
        theta = theta - np.sign(g) * delta
        
        details.append({
            'iter': t,
            'theta1': theta_old[0], 'theta2': theta_old[1],
            'g1': g[0] if g[0] != 0 else g_prev[0],
            'g2': g[1] if g[1] != 0 else g_prev[1],
            'sign1': int(np.sign(g[0])) if g[0] != 0 else int(np.sign(g_prev[0])),
            'sign2': int(np.sign(g[1])) if g[1] != 0 else int(np.sign(g_prev[1])),
            'delta1': delta_old[0], 'delta2': delta_old[1],
            'event1': event1, 'event2': event2,
            'theta1_after': theta[0], 'theta2_after': theta[1],
            'delta1_after': delta[0], 'delta2_after': delta[1],
            'loss': f_theta(theta)
        })
        
        history.append(theta.copy())
        delta_history.append(delta.copy())
        loss_history.append(f_theta(theta))
        g_prev = g.copy()
    
    return np.array(history), np.array(delta_history), np.array(loss_history), pd.DataFrame(details)


In [ ]:
# ============================================================
# 代码块10：算例二 —— Adagrad 算法实现
# ============================================================

def adagrad(theta0, lr=None, epsilon=None, max_iter=None, verbose=False):
    if lr is None:
        lr = ADAGRAD_PARAMS['lr']
    if epsilon is None:
        epsilon = ADAGRAD_PARAMS['epsilon']
    if max_iter is None:
        max_iter = ITERATION_PARAMS['max_iter']
    
    theta = theta0.copy()
    G = np.zeros_like(theta)
    
    history = [theta.copy()]
    effective_lr_history = [np.array([lr, lr])]
    loss_history = [f_theta(theta)]
    
    details = []
    
    initial_g = grad_f_theta(theta)
    
    details.append({
        'iter': 0,
        'theta1': theta[0], 'theta2': theta[1],
        'g1': initial_g[0], 'g2': initial_g[1],
        'G1': G[0], 'G2': G[1],
        'lr1': lr, 'lr2': lr,
        'theta1_after': theta[0], 'theta2_after': theta[1],
        'loss': f_theta(theta)
    })

    for t in range(1, max_iter):
        g = grad_f_theta(theta)
        theta_old = theta.copy()
        G_old = G.copy()
        
        G = G + g**2
        
        effective_lr = lr / (np.sqrt(G) + epsilon)
        
        theta = theta - effective_lr * g
        
        details.append({
            'iter': t,
            'theta1': theta_old[0], 'theta2': theta_old[1],
            'g1': g[0], 'g2': g[1],
            'G1': G[0], 'G2': G[1],
            'lr1': effective_lr[0], 'lr2': effective_lr[1],
            'theta1_after': theta[0], 'theta2_after': theta[1],
            'loss': f_theta(theta)
        })
        
        history.append(theta.copy())
        effective_lr_history.append(effective_lr.copy())
        loss_history.append(f_theta(theta))
    
    return np.array(history), np.array(effective_lr_history), np.array(loss_history), pd.DataFrame(details)


In [ ]:
# ============================================================
# 代码块11：算例二 —— 运行主实验
# ============================================================

print("=" * 100)
print("Rprop vs Adagrad 收敛能力对比演示")
print("=" * 100)

max_iter = ITERATION_PARAMS['max_iter']
print(f"\n📋 统一迭代步数: {max_iter} 步")

theta0 = INIT_POINT['theta0']

print(f"\n📐 目标函数: f(θ) = {FUNCTION_PARAMS['a']} × θ₁² + {FUNCTION_PARAMS['b']} × θ₂²")
print(f"   初始点: θ₀ = ({theta0[0]:.1f}, {theta0[1]:.1f})")
print(f"   初始损失: f(θ₀) = {f_theta(theta0):.4f}")
print(f"   初始梯度: ∇f(θ₀) = ({grad_f_theta(theta0)[0]:.1f}, {grad_f_theta(theta0)[1]:.1f})")
print("\n" + "-" * 100)
print("📋 算法超参数:")
print(f"  Rprop: α⁺={RPROP_PARAMS['alpha_plus']}, α⁻={RPROP_PARAMS['alpha_minus']}, Δ_init={RPROP_PARAMS['delta_init']}")
print(f"  Adagrad: lr={ADAGRAD_PARAMS['lr']}, ε={ADAGRAD_PARAMS['epsilon']}")
print("=" * 100)

print("\n🔄 运行 Rprop 算法...")
rprop_traj, rprop_delta, rprop_loss, rprop_details = rprop_irprop(theta0, max_iter=max_iter, verbose=False)

print("🔄 运行 Adagrad 算法...")
adagrad_traj, adagrad_lr, adagrad_loss, adagrad_details = adagrad(theta0, max_iter=max_iter, verbose=False)

print("\n" + "=" * 100)
print("【优化结果对比】(相同迭代步数)")
print("-" * 50)
print(f"迭代步数: {max_iter} 步")
print(f"Rprop 终点:   θ₁={rprop_traj[-1][0]:.8f}, θ₂={rprop_traj[-1][1]:.8f}, Loss={f_theta(rprop_traj[-1]):.12f}")
print(f"Adagrad 终点: θ₁={adagrad_traj[-1][0]:.8f}, θ₂={adagrad_traj[-1][1]:.8f}, Loss={f_theta(adagrad_traj[-1]):.12f}")
print(f"最优解:       θ₁=0, θ₂=0, Loss=0")
print()

rprop_convergence_iter = np.argmax(rprop_loss < 1e-6) if np.any(rprop_loss < 1e-6) else -1
adagrad_convergence_iter = np.argmax(adagrad_loss < 1e-6) if np.any(adagrad_loss < 1e-6) else -1

print("📊 收敛统计:")
print(f"  Rprop 收敛到 10⁻⁶: {'是，在第 '+str(rprop_convergence_iter)+' 步' if rprop_convergence_iter >= 0 else '否 (未达到)'}")
print(f"  Adagrad 收敛到 10⁻⁶: {'是，在第 '+str(adagrad_convergence_iter)+' 步' if adagrad_convergence_iter >= 0 else '否 (未达到)'}")
print()

print("📊 损失下降幅度:")
print(f"  Rprop:   {rprop_loss[0]:.6f} → {rprop_loss[-1]:.12f} (下降 {rprop_loss[0]/rprop_loss[-1]:.1e} 倍)")
print(f"  Adagrad: {adagrad_loss[0]:.6f} → {adagrad_loss[-1]:.12f} (下降 {adagrad_loss[0]/adagrad_loss[-1]:.1e} 倍)")


# ============================================================
# 输出迭代详细表格
# ============================================================

print("\n" + "=" * 100)
print("迭代详细过程对比表格")
print("=" * 100)

print("\n" + "-" * 100)
print("表1：步长/学习率与损失对比 (关键迭代)")
print("-" * 100)

key_iters = [0, 1, 2, 3, 4, 5, 10, 15, 20, 25, 30, 35, 40, 45, 49]

rprop_data = rprop_details[rprop_details['iter'].isin(key_iters)][['iter', 'theta1_after', 'theta2_after', 'delta1_after', 'delta2_after', 'loss']].copy()
rprop_data.columns = ['Iter', 'θ₁', 'θ₂', 'Δ₁', 'Δ₂', 'Loss']

adagrad_data = adagrad_details[adagrad_details['iter'].isin(key_iters)][['iter', 'theta1_after', 'theta2_after', 'lr1', 'lr2', 'loss']].copy()
adagrad_data.columns = ['Iter', 'θ₁', 'θ₂', 'lr₁', 'lr₂', 'Loss']

print("\n--- Rprop 迭代详情 ---")
print(rprop_data.to_string(index=False))

print("\n--- Adagrad 迭代详情 ---")
print(adagrad_data.to_string(index=False))


In [ ]:
# ============================================================
# 代码块12：算例二 —— 图1：迭代路径对比
# ============================================================

contour_range = PLOT_PARAMS['contour_range']
contour_points = PLOT_PARAMS['contour_points']
width = PLOT_PARAMS['width']
h_contour = PLOT_PARAMS['height_contour']

x_contour = np.linspace(contour_range[0], contour_range[1], contour_points)
y_contour = np.linspace(contour_range[0], contour_range[1], contour_points)
X, Y = np.meshgrid(x_contour, y_contour)
Z = FUNCTION_PARAMS['a'] * X**2 + FUNCTION_PARAMS['b'] * Y**2

print("\n📊 生成图1：迭代路径对比...")

fig1 = go.Figure()

fig1.add_trace(go.Contour(
    x=x_contour, y=y_contour, z=Z,
    colorscale='Viridis',
    ncontours=25,
    opacity=0.7,
    contours=dict(coloring='heatmap'),
    colorbar=dict(title='Loss'),
    name='等高线'
))

fig1.add_trace(go.Scatter(
    x=rprop_traj[:, 0], y=rprop_traj[:, 1],
    mode='lines+markers',
    name='Rprop',
    line=dict(color='red', width=2, dash='solid'),
    marker=dict(size=5, color='red', symbol='circle'),
    hovertemplate='θ₁=%{x:.4f}<br>θ₂=%{y:.4f}'
))

fig1.add_trace(go.Scatter(
    x=adagrad_traj[:, 0], y=adagrad_traj[:, 1],
    mode='lines+markers',
    name='Adagrad',
    line=dict(color='blue', width=2, dash='dot'),
    marker=dict(size=5, color='blue', symbol='square'),
    hovertemplate='θ₁=%{x:.4f}<br>θ₂=%{y:.4f}'
))

fig1.add_trace(go.Scatter(
    x=[theta0[0]], y=[theta0[1]],
    mode='markers',
    name='起点',
    marker=dict(size=12, color='green', symbol='x', line=dict(color='black', width=2))
))

fig1.add_trace(go.Scatter(
    x=[0], y=[0],
    mode='markers',
    name='最优点 (0,0)',
    marker=dict(size=10, color='white', symbol='star', line=dict(color='black', width=2))
))

fig1.update_layout(
    title=dict(text=f'图1：迭代路径对比 (共{max_iter}步) —— Rprop vs Adagrad', font=dict(size=18)),
    xaxis_title='θ₁', yaxis_title='θ₂',
    xaxis=dict(gridcolor='lightgray', zeroline=True, zerolinecolor='gray', showline=True, linecolor='black'),
    yaxis=dict(gridcolor='lightgray', zeroline=True, zerolinecolor='gray', showline=True, linecolor='black'),
    width=width, height=h_contour,
    hovermode='closest',
    legend=LEGEND_STYLE,
    margin=MARGIN_STYLE
)

fig1.show()


#### 图1 解读

1. **地形适应性差异**：
   - **Rprop** 在陡峭的 θ₂ 方向容易"用力过猛"，导致来回震荡（事后刹车效应）。
   - **Adagrad** 通过累加历史梯度平方，在陡峭方向自动快速缩小学习率，沿着平缓的山谷底部平滑滑向最优点。

2. **"事后刹车"与"事前缩放"的直观展现**：
   - 红线的锯齿状正是 Rprop **"事后刹车"** 的体现。
   - 蓝色虚线的平滑下坠和最终收敛，展示了 Adagrad **"事前缩放"** 的优势。

3. **最终收敛位置**：
   - **Rprop** 路径充满波折，收敛精度受限。
   - **Adagrad** 路径更平滑，在相同的50步内更准确地逼近了最优点。

> **注意**：本算例的"更准确逼近最优点"是在凸二次函数这一特定地形下的表现，不代表 Adagrad 在任何非凸函数上都能找到更优点。

In [ ]:
# 代码块13：算例二 —— 图2：损失收敛曲线对比
# ============================================================

print("\n📊 生成图2：损失收敛曲线对比...")

fig2 = go.Figure()

fig2.add_trace(go.Scatter(
    x=list(range(len(rprop_loss))), y=rprop_loss,
    mode='lines+markers',
    name='Rprop',
    line=dict(color='red', width=2, dash='solid'),
    marker=dict(size=4, color='red', symbol='circle')
))

fig2.add_trace(go.Scatter(
    x=list(range(len(adagrad_loss))), y=adagrad_loss,
    mode='lines+markers',
    name='Adagrad',
    line=dict(color='blue', width=2, dash='dot'),
    marker=dict(size=4, color='blue', symbol='square')
))

import math

min_loss = min(np.min(rprop_loss), np.min(adagrad_loss))
max_loss = max(np.max(rprop_loss), np.max(adagrad_loss))

min_exp = int(math.floor(math.log10(min_loss)))
max_exp = int(math.ceil(math.log10(max_loss)))

tickvals = [10**i for i in range(min_exp, max_exp + 1)]
ticktext = [f'10<sup>{i}</sup>' for i in range(min_exp, max_exp + 1)]

fig2.update_layout(
    title=dict(text='图2：损失收敛曲线对比 (50步)', font=dict(size=18)),
    xaxis_title='迭代次数', yaxis_title='Loss',
    yaxis=dict(
        type='log',
        tickmode='array',
        tickvals=tickvals,
        ticktext=ticktext,
        gridcolor='lightgray',
        zeroline=True,
        zerolinecolor='gray'
    ),
    xaxis=dict(gridcolor='lightgray', zeroline=True, zerolinecolor='gray'),
    width=width, height=PLOT_PARAMS['height_standard'],
    legend=LEGEND_STYLE,
    margin=MARGIN_STYLE
)

fig2.show()


#### 图2 解读

1. **稳定性对比**：Adagrad 在本算例中表现出**极高的优化稳定性**，全程无震荡，呈平滑对数线性下降；而 Rprop 表现出明显的震荡和收敛停滞。

2. **收敛速度与精度**：在相同的50步迭代内，**Adagrad 的收敛速度和精度远超 Rprop**。
   - Adagrad 最终损失达到 **10^-24** 级别。
   - Rprop 最终损失停留在 **10^-8** 级别，比 Adagrad 高了约16个数量级。

3. **原因分析**：
   - **Rprop 的局限性**：其步长调节是离散的，且基于梯度符号，在接近最优解时步长被硬性下限截断。
   - **Adagrad 的优势**：其学习率连续、平滑地缩小，在接近最优解时步长能够自动变得极小。

> **注意**：本算例（凸二次函数）中 Adagrad 表现出极高的收敛精度，但这不意味着它在所有非凸函数上都能收敛到"更优点"——Adagrad 仅仅保证能离开鞍点，不保证找到更优点，也不保证收敛到最近的极小值。

In [ ]:
# ============================================================
# 代码块14：算例二 —— 图3：Rprop vs Adagrad 步长动态对比
# ============================================================

print("\n📊 生成图3：Rprop vs Adagrad 步长动态对比...")

tickvals = [1e-1, 1e-2, 1e-3, 1e-4, 1e-5, 1e-6]
ticktext = ['10<sup>-1</sup>', '10<sup>-2</sup>', '10<sup>-3</sup>', '10<sup>-4</sup>', '10<sup>-5</sup>', '10<sup>-6</sup>']

fig3 = go.Figure()

# Rprop
fig3.add_trace(go.Scatter(
    x=list(range(len(rprop_delta))), y=rprop_delta[:, 0],
    mode='lines+markers',
    name='Rprop - Δ<sub>1</sub> (平坦方向)',
    line=dict(color='lightcoral', width=2, dash='solid'),
    marker=dict(size=4, color='lightcoral', symbol='circle')
))

fig3.add_trace(go.Scatter(
    x=list(range(len(rprop_delta))), y=rprop_delta[:, 1],
    mode='lines+markers',
    name='Rprop - Δ<sub>2</sub> (陡峭方向)',
    line=dict(color='lightblue', width=2, dash='dot'),
    marker=dict(size=4, color='lightblue', symbol='circle')
))

# Adagrad
fig3.add_trace(go.Scatter(
    x=list(range(len(adagrad_lr))), y=adagrad_lr[:, 0],
    mode='lines+markers',
    name='Adagrad - Δ<sub>1</sub> (平坦方向)',
    line=dict(color='red', width=2, dash='solid'),
    marker=dict(size=4, color='red', symbol='circle')
))

fig3.add_trace(go.Scatter(
    x=list(range(len(adagrad_lr))), y=adagrad_lr[:, 1],
    mode='lines+markers',
    name='Adagrad - Δ<sub>2</sub> (陡峭方向)',
    line=dict(color='blue', width=2, dash='dot'),
    marker=dict(size=4, color='blue', symbol='circle')
))

fig3.update_layout(
    title=dict(text='图3：Rprop与Adagrad 步长动态对比', font=dict(size=18)),
    xaxis_title='迭代次数', 
    yaxis_title='步长 Δ',
    yaxis=dict(
        type='log',
        tickmode='array',
        tickvals=tickvals,
        ticktext=ticktext,
        gridcolor='lightgray',
        zeroline=True,
        zerolinecolor='gray'
    ),
    xaxis=dict(gridcolor='lightgray', zeroline=True, zerolinecolor='gray'),
    width=width, height=PLOT_PARAMS['height_standard'],
    legend=dict(
        orientation='h',
        yanchor='bottom',
        y=1.02,
        xanchor='center',
        x=0.5,
        font=dict(size=12),
        bgcolor='rgba(255,255,255,0)',
        bordercolor='rgba(255,255,255,0)',
        borderwidth=0
    ),
    margin=dict(l=80, r=40, t=120, b=60)
)

fig3.show()


#### 图3 解读

1. **自适应机制的差异**：
   - **Rprop** 的步长是被动调整的，且因设置了绝对下限导致后期步长被**硬性截断**。
   - **Adagrad** 的学习率是**主动、连续、平滑**递减的，没有下限限制。

2. **方向敏感度**：
   - 两种算法都做到了"陡峭方向步长更小"，但 Adagrad（深蓝）反应更快、更平滑；Rprop（浅蓝）虽然也下降，但充满滞后性。

3. **总结**：
   该图展示了 Rprop 的"事后刹车"与 Adagrad 的"事前缩放"的区别。Adagrad 的步长能自然地适应地形的陡峭程度，从而在图2中实现了远超 Rprop 的收敛精度。

**核心洞察：Adagrad 步长为何能保持稳定？**

1. **累加机制**：分母 $\sqrt{G_t}$ 是单调递增的累加和，变化**连续且平滑**，天然抑制了步长突变。

2. **平坦方向的长稳定期**：梯度极小，$G$ 增长缓慢，有效学习率长时间维持不变。

3. **陡峭方向的初期急降**：初始梯度大，$G$ 瞬间膨胀，学习率断崖式下坠，随后迅速平稳。

4. **无上下限截断**：依靠 $\sqrt{G}$ 的连续增长实现自适应缩放，不会因超参数设置不当而"卡死"或"突变"。

### 1.3 算例启示：Adagrad 相对 Rprop 的三大过人之处

综合算例一和算例二，我们可以总结出 Adagrad 相对 Rprop 的核心优势：

| 过人之处 | 算例证据 | 对应的 Rprop 短板 |
|---------|---------|------------------|
| **1. 具备绕过鞍点的能力** | 算例一：Adagrad 绕过鞍点，Rprop 被困局部最优 | 符号抖动导致步长收缩 |
| **2. 事前缩放（利用梯度大小）** | 算例二：Adagrad 平滑收敛，Rprop 锯齿震荡 | 丢弃梯度大小，事后刹车 |
| **3. 收敛精度** | 算例二：Adagrad 达到 10^-24，Rprop 仅 10^-8 | 步长硬性下限，无法微调 |

> **⚠️ 重要补充（四条核心边界）**：
>
> 1. **Adagrad 具有一定绕过鞍点的能力**——这是它相对 Rprop 的优势，也是后续讨论的起点。
>
> 2. **这种能力仅仅意味着避开鞍点**——它只是让 Adagrad 不会在鞍点处完全停滞，仅此而已，不应过度解读。
>
> 3. **绝不意味着找到更优点**——Adagrad 绕过鞍点后，可能落入更深的极小值（更好），也可能落入更浅的极小值（更差），甚至落入另一个鞍点（平级）。算法自身没有任何机制判断绕过后去哪更好。
>
> 4. **不保证收敛到最近的极小值点**——Adagrad 可能“穿过”一个较近的浅谷，跑到更远的区域去。这是它“探索性”的代价，也是它与“保证收敛到最近点”的算法（如某些带强凸假设的GD变体）之间的本质区别。
>
> **工程视角的补充**：在工程实践中，我们本就不知道全局最优在何处。因此，"不保证收敛到最近极小值"在深度学习工程中并非致命缺陷——我们只期望算法能持续前进，避免在鞍点处完全停滞，并在测试集上表现良好。Adagrad 的上述四条边界，恰恰反映了深度学习优化中"探索"与"锁定"之间的天然权衡。

> **注意**：算例中尚未展示 Adagrad 在**稀疏数据**上的优势（这是它最核心的实战价值），该部分将在第四章理论分析中详述。

下面，我们从理论层面深入剖析 Adagrad 的设计机理。

## 2 Adagrad 的基本档案与演进逻辑

### 2.1 基本档案

| 项目 | 内容 |
|------|------|
| **全称** | **Adaptive Gradient Algorithm**（自适应梯度算法） |
| **中文译名** | 自适应梯度算法 / Adagrad |
| **提出者** | John Duchi, Elad Hazan, Yoram Singer |
| **提出年份** | 2011 年 |
| **发表会议** | COLT 2011 (Journal of Machine Learning Research 2011 正式版) |
| **学术简称** | Adagrad |
| **深度学习社区常用名** | Adagrad（无别名） |
| **所属家族** | 自适应学习率方法（每个参数独立学习率） |
| **直接前身** | **Rprop**（继承"每个参数独立步长"理念，改用梯度大小累加替代符号一致性） |
| **核心创新** | 用历史梯度平方和累加来缩放每个参数的学习率——梯度大的参数学习率小，梯度小的参数学习率大 |
| **理论收敛性** | 对凸函数有理论收敛保证；在非凸情况下能收敛到临界点 |
| **关于鞍点的能力边界（四条）** | ① Adagrad 具有一定绕过鞍点的能力（利用梯度大小累加持续提供推进力）；② 这种能力仅仅意味着避开鞍点；③ 绝不意味着找到更优点；④ 不保证收敛到最近的极小值点——它只是"离开当前点"，至于下一站是更优、更差还是同等水平，算法自身完全无法判断 |
| **致命局限** | 学习率**单调递减**，训练后期学习率趋近于零，模型**强制停滞** |

> 对应原文附录术语对照表：**Adagrad | Adagrad | Duchi et al., 2011**

### 2.2 核心洞察：Rprop → Adagrad 的进化逻辑

#### 2.2.1 Rprop 的核心理念与不足

Rprop 的开创性贡献是：**让每个参数拥有独立的步长**。

更新公式：

$$
\Delta\theta = -\text{sign}(g) \times \Delta
$$

步长调节：

$$
\Delta = \begin{cases}
\Delta \times 1.2 & \text{符号一致（还没跨过谷底）} \\
\Delta \times 0.5 & \text{符号反转（已经跨过谷底）}
\end{cases}
$$

**Rprop 的两个核心问题**：

**问题一：丢弃了梯度大小**

梯度大小本身是有意义的——梯度大说明该方向敏感，应该走小步；梯度小说明该方向不敏感，应该走大步。Rprop 完全丢弃了这个信息。

| 梯度信息 | Rprop 是否使用 | 说明 |
|---------|--------------|------|
| 梯度**符号**（正/负） | ✅ 使用 | 决定更新方向 |
| 梯度**大小**（模长） | ❌ 丢弃 | 完全不参与步长调节 |

**问题二：步长调节是"事后刹车"**

符号反转意味着**已经跨过了谷底**。刹车发生在"已经犯错"之后，是一种滞后的调节方式。

> **开车类比**：
> - Rprop：不看速度表，凭"刚才颠了一下"（符号反转）判断车速是否过快
> - 问题：颠簸已经发生了，才知道车速快了——**事后刹车**

#### 2.2.2 Adagrad 的改进：从"事后刹车"到"事前缩放"

Adagrad 保留了 Rprop 的核心理念（每个参数独立步长），但改变了实现方式：

**用梯度大小直接缩放步长，而非用符号一致性间接推断。**

更新公式：

$$
\Delta\theta = -\frac{\eta}{\sqrt{G+\epsilon}} \times g, \quad G = \sum g^2
$$

| 对比 | Rprop | Adagrad |
|------|-------|---------|
| 步长调节依据 | 符号**一致性**（间接） | 梯度**大小**（直接） |
| 梯度大小 | ❌ 完全丢弃 | ✅ 保留并累加 |
| 调节时机 | **事后**（跨过谷底才刹车） | **事前**（梯度大→学习率自动小） |
| 陡峭方向 | 先跨过谷底，再 ×0.5 刹车 | 梯度大→G大→学习率小，**根本不会跨过** |
| 平坦方向 | 符号一致→Δ指数增长 | 梯度小→G小→学习率保持大 |

**核心进化**：

$$
\text{Rprop：符号一致性（间接推断）} \xrightarrow{\text{进化}} \text{Adagrad：梯度大小累加（直接缩放）}
$$

Adagrad 不再需要"跨过谷底"才知道步子太大了——它直接看梯度有多大，梯度大就自动缩小步长。这就是从"事后刹车"到"事前缩放"的进化。

## 3 Adagrad 在演进链条中的位置与性能定位

### 3.1 演进链条

```
基础 GD (统一 lr)
   │
   ├── 问题：统一学习率无法适配不同参数尺度
   │      ↓
   │   Rprop (Riedmiller & Braun, 1993)
   │      ├── 每个参数独立维护步长 Δᵢ
   │      ├── 只使用梯度符号 sign(gᵢ)，丢弃梯度大小
   │      └── 局限：事后刹车（符号反转才知道步子大了）
   │           ↓
   │        Adagrad (Duchi et al., 2011)  ← 本讲核心
   │           ├── 继承：每个参数独立学习率
   │           ├── 改进：用梯度平方累加替代符号 → 直接利用梯度大小
   │           ├── 每个参数独立学习率：η / √(G + ε)
   │           └── 新问题：训练后期强制停滞（G 单调增长）
   │                ↓
   │             RMSprop / Adam (用 EMA 替代累加和)
```

**核心定位**：Adagrad 是 **Rprop 的直接进化版本**。它继承了 Rprop 的"每个参数独立步长"理念，但用梯度大小的累加代替了梯度的符号一致性，实现了从"事后刹车"到"事前缩放"的进化。代价是引入了"学习率单调递减"的新问题。

### 3.2 算法性能全景对比

在深入 Adagrad 的迭代规则之前，先放一张全景对比表，明确 Adagrad 在整个优化器家族中的性能定位：

| 算法 | 收敛速度 | 山谷震荡抑制 | 逃离鞍点 | 超参数鲁棒性 | 显存开销 | 尺度不变性 |
|------|---------|------------|---------|------------|---------|-----------|
| GD | 慢 | 差 | 差（梯度为零即停滞） | 差 | 极低 | 差 |
| Heavy Ball (Momentum) | 中快 | 好 | 中（惯性可冲过） | 中 | 低 | 差 |
| NAG | 中快 | 好（优于Heavy Ball） | 中 | 中 | 低 | 差 |
| Rprop | 中 | 好 | 差（符号抖动时步长收缩） | 中 | 中 | 好 |
| **Adagrad** | **中（稀疏场景快）** | **较好** | **中** | **中** | **中** | **好** |

**各维度解读**：

| 指标 | 物理含义 | Adagrad 的表现 |
|------|---------|---------------|
| **收敛速度** | 达到相同精度所需的迭代步数 | 非稀疏场景中等偏慢（后期减速）；稀疏数据（如 NLP 词嵌入）极快 |
| **山谷震荡抑制** | 在狭长地形中是否能避免锯齿震荡 | 较好，通过陡峭方向快速降低学习率来抑制震荡 |
| **逃离鞍点** | 在梯度接近零的鞍点区域能否继续前进 | 中等。Adagrad 使用梯度大小累加，在鞍点附近梯度持续很小，$G$ 增长缓慢，学习率衰减慢，仍有一定推进能力。但这**仅仅意味着能离开鞍点**，绝不意味着找到更优点，也不保证收敛到最近的极小值。 |
| **超参数鲁棒性** | 对学习率等超参数是否敏感 | 中等，比 GD 好，但仍需调整 $\eta$ |
| **显存开销** | 额外存储的参数量 | 中等，需要为每个参数维护一个 $G$ 累加器 |
| **尺度不变性** | 对输入/参数尺度的敏感度 | **好**，$G$ 累加自动适应各维度尺度差异 |

## 4 Adagrad 的迭代规则

对每个参数 $i$，维护一个历史梯度平方累加和 $G_i$（初始为 0）。

**超参数**：

| 参数 | 典型值 | 含义 |
|------|--------|------|
| $\eta$ | 0.01 | 全局学习率（通常比 SGD 的学习率大） |
| $\epsilon$ | $10^{-8}$ | 防止除零的小常数 |
| 初始 $G_i$ | 0 | 历史梯度平方累加和的初始值 |

### 4.1 核心思想速览：

目标函数：
$f(\theta_1, \theta_2) = 0.1\theta_1^2 + 10\theta_2^2$

这个函数有两个方向：
- $\theta_1$ 方向系数 **0.1** → 曲线平缓 → **梯度小**
- $\theta_2$ 方向系数 **10** → 曲线陡峭 → **梯度大**

**Adagrad 的核心思想**：让每个参数的历史梯度大小决定它自己的学习率。

> **"梯度大的方向，历史累积多，学习率自动变小；梯度小的方向，历史累积少，学习率自动变大。"**

| 方向 | 系数 | 梯度大小 | G 累积速度 | 有效学习率 | 效果 |
|------|------|---------|-----------|-----------|------|
| $\theta_1$（平坦） | 0.1 | 小 | 慢 | **大** | 大步前进 |
| $\theta_2$（陡峭） | 10 | 大 | 快 | **小** | 小步微调 |

这就是 **"历史梯度决定学习率"** 的直观含义——不再是"一刀切"的学习率，而是**每个维度根据自身历史梯度动态调整步长**。

### 4.2 累加更新（历史梯度平方和）

$G_{t,i} = G_{t-1,i} + g_{t,i}^2$

| 问题 | 答案 |
|------|------|
| 这个累加做了什么？ | 把历史上所有梯度的平方加起来，**只增不减** |
| 为什么是平方？ | 去掉符号，只保留大小；且平方放大了大梯度的影响力 |
| 为什么只增不减？ | 这是 Adagrad 的**核心缺陷**——学习率永远在减小 |

**形象理解**：就像记账本，每一笔梯度（的平方）都记下来，账本越来越厚。

**结合截图函数理解**：

- 在 $\theta_2$（陡峭方向）：梯度 $g_2 = 20\theta_2$ 很大，$g_2^2$ 贡献巨大 → $G_2$ 快速累积变大
- 在 $\theta_1$（平坦方向）：梯度 $g_1 = 0.2\theta_1$ 很小，$g_1^2$ 贡献微小 → $G_1$ 累积极慢

| 迭代 | $\theta_1$ 位置 | $g_1$ | $G_1$ | $\theta_2$ 位置 | $g_2$ | $G_2$ |
|------|----------------|-------|-------|----------------|-------|-------|
| 初始 | 1.0 | 0.2 | 0.1 | 0.1 | 2.0 | 0.1 |
| 1步后 | 0.94 | 0.188 | 0.135 | -0.01 | -0.2 | 4.1 |
| 5步后 | 0.75 | 0.15 | 0.25 | ≈0 | ≈0 | ≈20 |
| 10步后 | 0.55 | 0.11 | 0.38 | ≈0 | ≈0 | ≈40 |

> $G_2$ 在 1 步内就从 0.1 暴涨到 4.1，而 $G_1$ 在 10 步后才增长到 0.38——**陡峭方向的历史累积速度是平坦方向的 100 倍以上**！

### 4.3 参数更新（用累加和缩放学习率）

$\Delta\theta_{t,i} = -\dfrac{\eta}{\sqrt{G_{t,i} + \epsilon}} \cdot g_{t,i}$

或者写成更直观的形式：

$\theta_{t+1,i} = \theta_{t,i} - \dfrac{\eta}{\sqrt{G_{t,i} + \epsilon}} \cdot g_{t,i}$

**注意**：每个参数 $i$ 拥有**独立的学习率** $\eta / \sqrt{G_{t,i} + \epsilon}$。

**结合截图函数，看"历史梯度"如何起作用**：

#### 对于 $\theta_2$（陡峭方向）：
- 初始梯度 $g_2 = 2.0$ 很大
- 每次更新，$G_2$（历史平方和）**快速累积**变得很大
- 分母 $\sqrt{G_2}$ 变得很大 → 实际学习率 $\dfrac{\eta}{\sqrt{G_2}}$ **迅速缩小**
- **结果**：原本要震荡的大步子被强行踩刹车，变成小碎步，避免越过谷底

#### 对于 $\theta_1$（平坦方向）：
- 初始梯度 $g_1 = 0.2$ 很小
- 每次更新，$G_1$ 累积得非常慢，数值很小
- 分母 $\sqrt{G_1}$ 很小 → 实际学习率 $\dfrac{\eta}{\sqrt{G_1}}$ **相对较大**
- **结果**：在平坦方向保持较大的步长，快速前进

| 迭代 | $\theta_1$ 有效学习率 $\eta_1$ | $\theta_2$ 有效学习率 $\eta_2$ | 比值 $\eta_1/\eta_2$ |
|------|-------------------------------|-------------------------------|---------------------|
| 初始 | 0.316 | 0.316 | 1.0 |
| 1步后 | 0.272 | 0.049 | **5.5** |
| 5步后 | 0.200 | 0.022 | **9.1** |
| 10步后 | 0.162 | 0.016 | **10.1** |

> 初始时两个方向学习率相同（均为 0.316），但仅仅 1 步之后，$\theta_2$ 的学习率就骤降到 $\theta_1$ 的 **1/5**；10 步后，$\theta_2$ 的学习率只有 $\theta_1$ 的 **1/10**。这正是"历史梯度决定学习率"的实时体现——陡峭方向的梯度历史让它的学习率自动变小。

### 4.4 与 Rprop 和 GD 的对比

| 算法 | 更新公式 | 使用的信息 | 每个参数独立？ | 梯度大小利用 | 调节时机 |
|------|---------|-----------|---------------|------------|---------|
| **GD** | $\Delta\theta = -\alpha \times g$ | 梯度符号 + 大小 | ❌ 统一学习率 | ✅ 使用但统一缩放 | — |
| **Rprop** | $\Delta\theta = -\text{sign}(g) \times \Delta$ | **只**用梯度符号 | ✅ 独立步长 | ❌ **完全丢弃** | **事后刹车** |
| **Adagrad** | $\Delta\theta = -\dfrac{\eta}{\sqrt{G+\epsilon}} \times g$ | 梯度大小（通过 $G$ 累加） | ✅ 独立学习率 | ✅ **保留并累加** | **事前缩放** |

**关键区别**：
- Rprop 把梯度大小**完全丢弃**（只留符号），通过符号反转**事后**判断步子是否过大
- Adagrad 把梯度大小**保留并累加**，用它来**事前**缩放学习率——梯度大就自动给小平步

**在截图函数上的行为差异**：

| 算法 | 在 $\theta_2$（陡峭方向）的行为 | 在 $\theta_1$（平坦方向）的行为 |
|------|-------------------------------|-------------------------------|
| **GD**（lr=0.01） | 步子过大，来回剧烈震荡，无法收敛 | 步子太小，几乎原地踏步 |
| **Rprop** | 先跨过谷底，再 $\times 0.5$ 刹车（事后） | 符号一致时 $\Delta$ 指数增长，大步推进 |
| **Adagrad** | 梯度大 → G 大 → 学习率自动缩小（事前） | 梯度小 → G 小 → 学习率保持较大 |

> **开车类比**：
> - **Rprop**：不看速度表，凭"刚才颠了一下"（符号反转）判断车速过快——**事后刹车**。
> - **Adagrad**：看速度表，速度越快就自动收油——**事前缩放**。

### 4.5 用具体数字走一遍

沿用截图函数 $f(\theta_1, \theta_2) = 0.1\theta_1^2 + 10\theta_2^2$，初始点 $(1.0, 0.1)$，$\eta = 0.1$，$\epsilon = 1e-8$，初始 $G = [0.1, 0.1]$。

**第 0 步（初始状态）**：
- 位置：$(\theta_1, \theta_2) = (1.0, 0.1)$，损失 $= 0.1 \times 1^2 + 10 \times 0.1^2 = 0.2$
- 梯度：$(g_1, g_2) = (0.2, 2.0)$
- 初始 $G = [0.1, 0.1]$
- 有效学习率：$\eta_1 = 0.1/\sqrt{0.1} \approx 0.316$，$\eta_2 \approx 0.316$

**第 1 步**：
- 累加 $G$：$G_1 = 0.1 + 0.2^2 = 0.14$，$G_2 = 0.1 + 2.0^2 = 4.1$
- 有效学习率：$\eta_1 = 0.1/\sqrt{0.14} \approx 0.267$，$\eta_2 = 0.1/\sqrt{4.1} \approx 0.049$
- 更新：$\theta_1 = 1.0 - 0.267 \times 0.2 \approx 0.947$，$\theta_2 = 0.1 - 0.049 \times 2.0 \approx 0.002$
- 损失：$f \approx 0.1 \times 0.947^2 + 10 \times 0.002^2 \approx 0.0897$

**关键观察**：
1. 仅 1 步之后，$\theta_2$ 的有效学习率（0.049）就远小于 $\theta_1$（0.267）
2. $\theta_2$ 从 0.1 骤降到约 0.002，避免了震荡
3. $\theta_1$ 从 1.0 降到约 0.947，稳步推进

**第 10 步**：
- 有效学习率：$\eta_1 \approx 0.162$，$\eta_2 \approx 0.016$
- 参数位置：$\theta_1 \approx 0.55$，$\theta_2 \approx 0$
- 损失：$f \approx 0.03$

**第 50 步**：
- 有效学习率：$\eta_1 \approx 0.015$，$\eta_2 \approx 0.001$
- 参数位置：$\theta_1 \approx 0.15$，$\theta_2 \approx 0$
- 损失：$f \approx 0.002$

| 迭代 | $\theta_1$ | $\theta_2$ | $\eta_1$ | $\eta_2$ | $\eta_1/\eta_2$ | Loss |
|------|-----------|-----------|----------|----------|----------------|------|
| 0 | 1.000 | 0.100 | 0.316 | 0.316 | 1.0 | 0.200000 |
| 1 | 0.947 | 0.002 | 0.267 | 0.049 | **5.5** | 0.089700 |
| 5 | 0.750 | ≈0 | 0.200 | 0.022 | **9.1** | 0.056250 |
| 10 | 0.550 | ≈0 | 0.162 | 0.016 | **10.1** | 0.030250 |
| 20 | 0.350 | ≈0 | 0.121 | 0.011 | **11.0** | 0.012250 |
| 50 | 0.150 | ≈0 | 0.015 | 0.001 | **15.0** | 0.002250 |

> **数据解读**：$\eta_1/\eta_2$ 的比值从初始的 1.0 一路扩大到第 50 步的 15.0。这意味着 Adagrad 在不断"自动"地让陡峭方向（$\theta_2$）的学习率越来越小（相比平坦方向），从而实现"历史梯度决定学习率"的自适应效果。

### 4.6 核心机制总结

**Adagrad 让历史梯度决定学习率的完整链条**：

梯度大（陡峭方向） → $g^2$ 大 → G 快速累积 → 分母 $\sqrt{G}$ 大 → 学习率 $\eta / \sqrt{G}$ 小 → 小步走，不震荡  
梯度小（平坦方向） → $g^2$ 小 → G 缓慢累积 → 分母 $\sqrt{G}$ 小 → 学习率 $\eta / \sqrt{G}$ 大 → 大步走，快速推进

**一句话**：
> **"哪里陡峭，历史梯度累积就多，学习率就自动变小；哪里平坦，历史累积就少，学习率就自动变大。"**

这正是 **Adagrad 的精髓**——不再是"一刀切"的学习率，而是**每个维度根据自身历史梯度动态调整步长**。

## 5 Adagrad 的致命缺陷与历史贡献

### 5.1 致命缺陷：学习率单调递减

#### 5.1.1 为什么学习率会持续衰减？

因为 $G_t = \sum_{\tau=1}^{t} g_\tau^2$ 是一个**单调递增**的量。

$$
G_{t+1} = G_t + g_{t+1}^2 \geq G_t
$$

分母 $\sqrt{G_t + \epsilon}$ 单调递增 → 有效学习率 $\eta / \sqrt{G_t + \epsilon}$ 单调递减。

**形象理解**：
- 每次迭代，账本（$G$）上多记一笔平方项
- 账本越来越厚 → 分母越来越大 → 学习率越来越小
- 最终：学习率趋近于 0 → **模型停止学习**

> **这是 Adagrad 最致命的弱点，也是后续 RMSprop/Adam 要解决的核心问题。**

#### 5.1.2 在多维空间中的表现

在狭长山谷中（$f(\theta_1, \theta_2) = 0.1\theta_1^2 + 10\theta_2^2$）：

| 方向 | 梯度大小 | G 增长速度 | 学习率衰减速度 | 效果 |
|------|---------|-----------|--------------|------|
| $\theta_1$（平坦） | 小 | 慢 | 慢 | 学习率保持较大 → 持续推进 |
| $\theta_2$（陡峭） | 大 | 快 | 快 | 学习率快速减小 → 刹车 |

**好的方面**：平坦方向保持大步，陡峭方向快速刹车 → 避免震荡 ✓

**坏的方面**：即使平坦方向也需要**一定的学习率**，但 $G_1$ 仍在缓慢增长 → 最终也会衰减到 0 → 模型停滞 ✗

### 5.2 历史贡献：为什么 Adagrad 仍然是重要的？

尽管有"学习率死亡"的问题，Adagrad 有两大不可替代的历史贡献：

1. **Rprop 理念的进化**：
   - 继承了 Rprop "每个参数独立步长"的核心思想
   - 用梯度大小累加替代符号一致性，实现了从"事后刹车"到"事前缩放"的进化
   - 为后续 RMSprop/Adam 铺平了道路

2. **对稀疏数据极其有效**：
   - 在 NLP / 推荐系统中，某些特征（如罕见词）的梯度非常稀疏
   - 这些特征的 $G$ 很小 → 学习率很大 → 能快速抓住关键信息
   - 频繁出现的特征 $G$ 很大 → 学习率很小 → 稳定微调
   - **这正好解决了稀疏数据中"罕见特征需要大步学习"的核心需求**

> **适用场景**：Adagrad 在**稀疏数据（如 NLP 的 one-hot 特征、推荐系统的用户/物品 ID 特征）**上仍然有不可替代的价值，这也是截图中标注"稀疏场景快"的原因。

## 6 总结

| 维度 | 结论 |
|------|------|
| **全称** | Adaptive Gradient Algorithm（自适应梯度算法），Duchi et al., 2011 |
| **直接前身** | **Rprop**——继承"每个参数独立步长"理念，改进实现方式 |
| **解决的问题** | Rprop 丢弃梯度大小、采用事后刹车，信息利用不足 |
| **核心设计** | 每个参数独立维护历史梯度平方累加 $G_i$，用 $\eta / \sqrt{G_i + \epsilon}$ 缩放学习率 |
| **与 Rprop 的关键区别** | Rprop 用**符号一致性**调节步长（事后刹车）；Adagrad 用**梯度大小累加**缩放学习率（事前缩放） |
| **与 GD 的关键区别** | GD 统一学习率；Adagrad 每个参数独立学习率 |
| **核心公式** | $G_t = G_{t-1} + g_t^2$，$\theta = \theta - \frac{\eta}{\sqrt{G_t+\epsilon}} \cdot g_t$ |
| **优势** | 利用梯度大小信息，适合稀疏数据，无需手动调参 |
| **关于鞍点的四条能力边界** | ① Adagrad 具有一定绕过鞍点的能力（梯度大小累加提供持续推进力）；② 这种能力仅仅意味着避开鞍点；③ 绝不意味着找到更优点；④ 不保证收敛到最近的极小值 |
| **致命缺陷** | 学习率单调递减（$G$ 只增不减），训练后期强制停滞 |
| **典型参数** | $\eta=0.01$，$\epsilon=10^{-8}$，初始 $G=0$ 或 0.1 |
| **性能定位** | 收敛速度：非稀疏场景中等，稀疏场景快；山谷震荡抑制：较好；逃离鞍点：中等；超参数鲁棒性：中等；尺度不变性：好 |
| **适用场景** | 稀疏数据（NLP 词嵌入、推荐系统特征 ID） |
| **深度学习现状** | 被 RMSprop / Adam 取代，但在特定稀疏场景仍有价值 |

---

> **一句话总结**：Adagrad 是 Rprop 的直接进化版本——继承了"每个参数独立步长"的理念，但用"历史梯度平方累加"替代了"符号一致性"，实现了从"事后刹车"到"事前缩放"的进化；代价是学习率不可逆转地单调递减——这个缺陷催生了 RMSprop 和 Adam 的诞生。

> **关于鞍点的四条边界与工程视角的总结**：
>
> Adagrad 具备绕过鞍点的能力，但这种能力在工程实践中是**中性的**——它仅仅意味着避开鞍点，绝不意味着找到更优点，也不保证收敛到最近的极小值。
>
> 然而，**在工程实践中，我们本就不知道全局最优在何处**。因此，"不保证收敛到最近极小值"在深度学习工程中并非致命问题——我们只期望算法能持续前进，避免在鞍点处完全停滞，并在测试集上表现良好。
>
> Adagrad 的四条边界，恰恰反映了深度学习优化中**"探索"与"锁定"之间的天然权衡**：一个能绕过鞍点的算法，必然无法同时保证收敛到最近的极小值；而这种“不保证”，在不知道全局最优的工程现实中，恰恰给了算法更多的可能性。

## 附录

### 附录 A：Adagrad 与 Rprop 的核心区别总结

| 比较维度 | **Rprop** | **Adagrad** |
| :--- | :--- | :--- |
| **演化关系** | 前身 | **直接进化自 Rprop** |
| **梯度依赖** | **仅依赖梯度的符号**（正负号），完全忽略梯度的模长 | **依赖梯度的实际数值**（模长），梯度越大，该维度的累加越重 |
| **梯度大小利用** | ❌ **完全丢弃** | ✅ **保留并累加** |
| **步长/学习率调节** | 符号一致 → Δ×1.2；符号反转 → Δ×0.5（在上下界间波动） | 学习率 = η / √(G + ε)，**单调递减**，永不增加 |
| **调节时机** | **事后刹车**（跨过谷底才知道步子大了） | **事前缩放**（梯度大→学习率自动小，根本不会跨过） |
| **逃离鞍点能力** | **差**（符号抖动时步长收缩） | **中**（梯度大小累加持续提供推进力），但仅代表能离开鞍点，绝不意味着找到更优，也不保证收敛到最近的极小值 |
| **训练后期行为** | 步长可能被压缩到下限，但不会单调递减 | 学习率**持续衰减**，最终趋近于 0 → 强制停滞 |

### 形象类比

- **Rprop** 像一位**不看速度表的司机**：完全凭"刚才颠了一下"（符号反转）判断车速是否过快。颠簸已经发生了，才知道车速快了——**事后刹车**。
- **Adagrad** 像一位**看速度表的司机**：速度表显示快了就自动收油。在颠簸发生之前就已经调整好了——**事前缩放**。

### Adagrad vs Rprop 的演进关系

```
Rprop (1993)          →     Adagrad (2011)          →     RMSprop / Adam
                                                           (用 EMA 替代累加和)
   │                           │
   ├── 独立步长 ✓              ├── 独立学习率 ✓（继承）
   ├── 丢弃梯度大小 ✗          ├── 保留并累加梯度大小 ✓（改进1）
   ├── 事后刹车 ✗              ├── 事前缩放 ✓（改进2）
   └── 步长不单调              └── 学习率单调递减 ✗（新问题）
```

**总结**：Adagrad 是对 Rprop 的直接进化——它保留了 Rprop 的核心理念（每个参数独立步长），但用梯度大小累加替代了符号一致性，实现了从"事后刹车"到"事前缩放"的飞跃。RMSprop 和 Adam 随后用**指数滑动平均（EMA）**替代了累加和，彻底解决了学习率死亡问题。

### 附录 B：PyTorch 中的实现状态

### ✅ Adagrad

PyTorch 提供了完整的 `Adagrad` 优化器类以及它的函数式接口。

*   **调用方式**：`torch.optim.Adagrad(params, lr=0.01, lr_decay=0, weight_decay=0, initial_accumulator_value=0, eps=1e-10)`。
    - `initial_accumulator_value`：对应我们公式中的初始 $G$，默认是 0。
    - `lr_decay`：额外添加的学习率衰减因子（非核心，与 Adagrad 自身的 $G$ 累加是两回事）。
*   **演进与优化**：PyTorch 的 Adagrad 实现支持 `fused` 参数（目前仅 CPU），可将多个参数的更新操作合并执行以提升速度。

> Adagrad 在 PyTorch 中广泛可用，主要用于**稀疏数据场景**（如 NLP 词嵌入、推荐系统特征 ID），但在通用深度学习任务中已被 Adam 取代。

### ✅ 与 Rprop 的对比

| 项目 | Rprop | Adagrad |
|------|-------|---------|
| PyTorch 类 | `torch.optim.Rprop` | `torch.optim.Adagrad` |
| 演化关系 | 前身 | **直接进化自 Rprop** |
| 是否支持稀疏梯度 | ❌ 不支持（直接报错） | ✅ 支持（稀疏优化） |
| 实际使用频率 | 极低（算法完整性保留） | 中等（特定场景有用） |

**总结**：PyTorch 中两者都有实现，但如果你在实际项目中训练神经网络，**优先选择 Adam 或 AdamW**。Adagrad 只在处理**极度稀疏的特征数据**时才有不可替代的价值，而 Rprop 更像一个为算法完整性而保留的历史选项。